In [1]:
import numpy as np
import torch
import polars as pl

In [2]:
from deep_parity.model import MLP

/Users/dashiell/workspace/deep-parity/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [25]:
n = 10
model = MLP(n, 256, 512)

In [26]:
import numpy as np
from deep_parity.boolean_cube import generate_all_binary_arrays, fourier_transform

all_binary_arrays = torch.from_numpy(generate_all_binary_arrays(n)).to(torch.float32)
all_parities = all_binary_arrays.sum(dim=1) % 2

In [27]:
logits, cache = model.run_with_cache(all_binary_arrays)

In [28]:
linear_preacts = cache['hook_linear']

In [29]:
ft = fourier_transform(linear_preacts.T)

In [34]:
ft.shape

torch.Size([512, 1024])

In [32]:
linear_preacts.shape

torch.Size([1024, 512])

In [31]:
2 ** n

1024

In [36]:
ft.shape
linear_df = pl.DataFrame(ft.T.numpy(), schema=[str(i) for i in range(ft.shape[0])])

In [38]:
base_df = pl.DataFrame({
    'bits': all_binary_arrays.numpy(), 
    'parities': all_parities.numpy(), 
    'degree': all_binary_arrays.sum(dim=1).numpy()
}).with_columns(indices=pl.col('bits').arr.to_list().list.eval(pl.arg_where(pl.element() == 1)))

In [39]:
data = pl.concat([base_df, linear_df], how='horizontal')

In [46]:
data.head()

bits,parities,degree,indices,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,…,475,476,477,478,479,480,481,482,483,484,485,486,487,488,489,490,491,492,493,494,495,496,497,498,499,500,501,502,503,504,505,506,507,508,509,510,511
"array[f32, 10]",f32,f32,list[u32],f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,…,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
"[0.0, 0.0, … 0.0]",0.0,0.0,[],4.792341,8.742755,-3.571704,-4.561469,1.5964,1.57336,-1.504783,0.841798,2.074499,-2.413394,4.257338,-5.195476,-3.946187,-9.761691,2.941171,4.720051,-4.847284,-12.338072,1.157266,1.072647,-6.299317,2.137212,6.996746,0.223996,-0.095116,2.882404,4.656972,7.010179,6.019701,-1.586935,-2.913243,8.855089,3.074979,…,-0.068103,2.901556,-4.945755,3.005411,-1.497864,4.472789,1.619014,-4.074386,6.411877,4.004467,7.541836,6.51852,6.478396,-4.299721,8.015448,1.641307,-1.10354,-5.594541,-1.421119,9.008275,0.755662,4.098518,-13.896557,-2.797248,-5.181099,-0.817573,0.569706,-2.378841,0.293791,6.084093,9.464976,-5.567934,11.706028,-1.277706,8.795528,6.812182,-0.072817
"[0.0, 0.0, … 1.0]",1.0,1.0,[9],-0.700889,-0.203499,-0.085415,1.486703,-0.422416,0.390541,-0.250597,1.174473,-3.370224,-0.477617,-0.270915,-0.292407,0.05918,2.147251,-1.430612,1.8677,-0.26171,2.281429,3.229279,-0.177188,2.355791,-0.559007,-0.071075,-1.092057,1.540353,-1.318204,-1.511159,0.16394,-1.364453,1.848573,0.312137,-1.825719,-0.120052,…,-2.878758,0.080586,0.975575,0.704436,-2.335425,-0.35367,1.290114,-1.263608,-0.471955,-1.686726,0.408112,-0.867448,0.814143,2.03818,0.081011,1.73811,2.566427,-0.536016,0.637883,2.303666,1.145213,0.644864,1.191432,-2.829496,-0.337946,0.207907,-1.123974,-0.681035,0.718581,0.252816,1.595561,-2.503472,-0.831924,0.338682,-0.0546,-1.574025,-1.356707
"[0.0, 0.0, … 0.0]",1.0,1.0,[8],-0.831905,-0.247869,0.095511,1.331138,0.773353,-3.376291,-0.780729,-1.392518,1.262918,-0.61777,-1.764848,1.935428,1.508197,1.55541,-0.775125,-2.761628,0.250585,4.65166,-0.080491,-0.829046,1.933371,-0.678325,0.826581,1.113637,0.725861,-1.078171,1.603476,0.024073,0.819388,1.059723,4.041817,-1.233522,0.985469,…,-1.110297,-1.787168,0.740619,1.519748,0.555081,-4.271544,1.826687,0.207122,-1.610199,-0.828887,-2.058326,-0.263558,-0.448255,-0.331615,-1.549262,-0.088261,0.634913,1.238182,0.905899,-1.713616,-1.413258,-0.996971,1.547458,1.102983,0.464635,-1.626723,1.419534,-1.24238,-1.486231,-1.912393,-0.440527,1.125496,-3.353304,0.905718,-1.241576,0.091067,-0.057909
"[0.0, 0.0, … 1.0]",0.0,2.0,"[8, 9]",-5.4715e-9,4.8225e-8,-8.0909e-9,1.7928e-8,6.3446e-9,4.6566e-10,-3.2422e-8,-7.3807e-8,1.3039e-8,8.1724e-8,-5.6112e-8,8.5507e-8,7.7373e-8,1.3039e-7,1.6298e-9,9.1502e-8,9.3307e-8,-2.1700e-7,-4.9826e-8,9.9186e-8,2.5611e-8,1.7229e-8,7.9162e-9,-8.0792e-8,1.2340e-8,-3.0268e-8,-2.3982e-8,1.4537e-7,5.5879e-9,-2.3516e-8,-2.0955e-9,2.5262e-7,-2.1231e-8,…,-4.3306e-8,-2.6703e-8,-2.9919e-8,-9.3132e-8,-1.9977e-7,-7.3342e-9,-7.5437e-8,-7.4971e-8,2.7881e-8,2.5611e-9,-1.6938e-8,-9.2085e-8,1.0966e-7,-1.1176e-8,6.8365e-8,5.5879e-9,-7.1246e-8,-1.0815e-7,6.7521e-9,1.1642e-8,5.0059e-8,2.1653e-8,-8.4983e-8,1.0245e-8,4.5751e-8,1.5076e-8,6.7521e-8,-4.5286e-8,1.4668e-8,-8.8126e-8,6.4960e-8,-4.9826e-8,6.4028e-8,3.6671e-8,2.3938e-9,9.8255e-8,2.9802e-8
"[0.0, 0.0, … 0.0]",1.0,1.0,[7],1.346855,-3.267175,1.809966,1.190491,1.439774,1.589015,-2.494148,1.74326,-0.401547,-2.188394,0.166995,-0.309339,1.1538,1.103178,1.410643,-2.318562,0.163853,-0.323163,-1.543699,-0.102388,-1.138524,-1.591093,-0.023038,0.211522,-1.264374,-1.208969,0.358339,-1.539717,-0.260995,-0.995863,-2.593095,1.116819,0.330369,…,1.221084,-1.750995,-1.502585,-2.553746,-1.243326,0.199269,-4.186768,-1.419756,0.325709,-0.381066,0.86721,-1.120417,0.394486,3.393852,-2.199876,-1.174516,-2.757999,-0.361579,-2.542652,-1.13489,-0.95552,-1.611486,4.064225,1.

In [48]:
data.select(pl.exclude('bits', 'parities', 'indices')).unpivot(index='degree').group_by('degree', 'variable').agg(pl.col('value').pow(2).sum())

degree,variable,value
f32,str,f32
3.0,"""129""",1.1206e-12
10.0,"""395""",2.9747e-15
8.0,"""78""",1.4488e-13
10.0,"""132""",6.9940e-15
10.0,"""334""",1.1026e-14
…,…,…
4.0,"""273""",8.7804e-13
8.0,"""63""",1.0069e-13
4.0,"""206""",2.2718e-12


In [22]:
base_df.with_columns(indices=pl.col('bits').arr.to_list().list.eval(pl.arg_where(pl.element() == 1)))

bits,parities,degree,indices
"array[f32, 20]",f32,f32,list[u32]
"[0.0, 0.0, … 0.0]",0.0,0.0,[]
"[0.0, 0.0, … 1.0]",1.0,1.0,[19]
"[0.0, 0.0, … 0.0]",1.0,1.0,[18]
"[0.0, 0.0, … 1.0]",0.0,2.0,"[18, 19]"
"[0.0, 0.0, … 0.0]",1.0,1.0,[17]
…,…,…,…
"[1.0, 1.0, … 1.0]",1.0,19.0,"[0, 1, … 19]"
"[1.0, 1.0, … 0.0]",0.0,18.0,"[0, 1, … 17]"
"[1.0, 1.0, … 1.0]",1.0,19.0,"[0, 1, … 19]"


In [24]:
def make_base_parity_dataframe(n):
    all_binary_data = generate_all_binary_arrays(n)
    all_degrees = all_binary_data.sum(axis=1)
    all_parities = all_degrees % 2


    base_df = pl.DataFrame({
        'bits': all_binary_data, 
        'parities': all_parities, 
        'degree': all_degrees
    })
    base_df = base_df.with_columns(
        indices=pl.col('bits').arr.to_list().list.eval(pl.arg_where(pl.element() == 1))
    )
    return base_df


make_base_parity_dataframe(10).head()

bits,parities,degree,indices
"array[u8, 10]",u64,u64,list[u32]
"[0, 0, … 0]",0,0,[]
"[0, 0, … 1]",1,1,[9]
"[0, 0, … 0]",1,1,[8]
"[0, 0, … 1]",0,2,"[8, 9]"
"[0, 0, … 0]",1,1,[7]


In [ ]:
def make_base_parity_dataframe(n):
    all_binary_data = generate_all_binary_arrays(n)
    all_degrees = all_binary_data.sum(axis=1)
    all_parities = all_degrees % 2


    base_df = pl.DataFrame({
        'bits': all_binary_data, 
        'parities': all_parities, 
        'degree': all_degrees
    })
    base_df = base_df.with_columns(
        indices=pl.col('bits').arr.to_list().list.eval(pl.arg_where(pl.element() == 1))
    )
    return base_df


def calc_power_contributions(tensor, n):
    base_df = make_base_parity_dataframe(n)
    ft = fourier_transform(tensor)
    linear_df = pl.DataFrame(
        ft.T.numpy(),
        schema=[str(i) for i in range(ft.shape[0])]
    )
    data = pl.concat([base_df, linear_df], how='horizontal')

    power_vals = torch.cat([power_contribs[irrep].unsqueeze(0) for irrep in irreps], dim=0)
    val_data = pl.DataFrame(power_vals.detach().cpu().numpy(), schema=[f'dim{i}' for i in range(power_vals.shape[1])])

    return val_data, power_contribs


def fourier_analysis(model, n, epoch):
    W = model.linear.weight
    embeds = model.embed.weight.T
    embed_dim = embeds.shape[0]
    linear = (W @ embeds).T
    embed_power_df = calc_power_contributions(linear, n).with_columns(pl.lit(epoch).alias('epoch'))
    return embed_power_df



In [2]:
def generate_binary_array_efficient(n):
    # Create an array of all possible numbers for n bits
    numbers = np.arange(2**n, dtype=np.uint32)
    
    # Use broadcasting with a single bitwise operation
    return ((numbers[:, np.newaxis] >> np.arange(n)[::-1]) & 1).astype(np.uint8)


generate_binary_array_efficient(25)

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 1],
       [0, 0, 0, ..., 0, 1, 0],
       ...,
       [1, 1, 1, ..., 1, 0, 1],
       [1, 1, 1, ..., 1, 1, 0],
       [1, 1, 1, ..., 1, 1, 1]], dtype=uint8)